In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')

In [17]:
df = pd.read_csv("../data/processed/hr_data_engineered.csv")

print(f"Dataset loaded: {df.shape}")
print("\nTarget distribution:")
print(df['target'].value_counts())


Dataset loaded: (50124, 51)

Target distribution:
target
0    34950
1    15174
Name: count, dtype: int64


In [18]:
X = df.drop('target', axis=1)
y = df['target']

# Safety: only numeric features
X = X.select_dtypes(include=[np.number])

print("Features:", X.shape)
print("Target:", y.shape)


Features: (50124, 31)
Target: (50124,)


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train class distribution:", np.bincount(y_train))
print("Test class distribution :", np.bincount(y_test))


Train class distribution: [27960 12139]
Test class distribution : [6990 3035]


In [20]:
log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'  # handle imbalance safely
)

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]


In [21]:
print("Logistic Regression Performance")
print("--------------------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Logistic Regression Performance
--------------------------------
Accuracy : 0.8587531172069826
Precision: 0.7175490459553884
Recall   : 0.8797364085667215
F1 Score : 0.7904085257548845
ROC-AUC  : 0.9430616107265499

Confusion Matrix:
[[5939 1051]
 [ 365 2670]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.85      0.89      6990
           1       0.72      0.88      0.79      3035

    accuracy                           0.86     10025
   macro avg       0.83      0.86      0.84     10025
weighted avg       0.87      0.86      0.86     10025



In [22]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]


In [23]:
print("\nRandom Forest Performance")
print("--------------------------")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test, rf_prob))



Random Forest Performance
--------------------------
Accuracy : 0.8890773067331671
Precision: 0.7990668740279938
Recall   : 0.8464579901153213
F1 Score : 0.82208
ROC-AUC  : 0.9554042607349166


In [24]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, rf_pred)
    ],
    'ROC_AUC': [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

print(results)


                 Model  Accuracy   ROC_AUC
0  Logistic Regression  0.858753  0.943062
1        Random Forest  0.889077  0.955404


In [25]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values(by='importance', ascending=False)

feature_importance.head(10)


,feature,importance
5,technical_score,0.153824
30,exp_salary_ratio,0.107562
26,interview_score,0.100367
27,total_score,0.076583
11,years_of_experience,0.063789
23,overall_assessment,0.058986
8,skills_match_percentage,0.052191
16,job_role_match,0.047937
29,skills_tech_ratio,0.037217
24,ctc_gap,0.033126


In [26]:
import pickle

best_model = rf_model  # or log_model if mentor prefers explainability

with open("../models/best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("model saved successfully")


model saved successfully


In [27]:
print("\n FINAL ML SUMMARY ")
print("• Problem Type        : Binary Classification")
print("• Models Used         : Logistic Regression, Random Forest")
print("• Imbalance Handling  : Class Weighting")
print("• Key Metric          : ROC-AUC")
print("• Business Focus      : Interpretability + Accuracy")
print("• Status              : Model ready for deployment")



 FINAL ML SUMMARY 
• Problem Type        : Binary Classification
• Models Used         : Logistic Regression, Random Forest
• Imbalance Handling  : Class Weighting
• Key Metric          : ROC-AUC
• Business Focus      : Interpretability + Accuracy
• Status              : Model ready for deployment


In [28]:
# Save feature names for inference
import pickle

with open("../models/model_features.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)